# 1. Data Foundation — What Can the Data Safely Tell Us?

Before looking for a story, we preserve the raw tables and create separate
cleaned working copies.

- `customers` — one row per customer ID
- `product_usage` — one row per customer-product-month
- `support_tickets` — one row per ticket

**Why clean and audit first?** Incorrect keys or joins can manufacture a
convincing but false relationship. A raw ticket-to-usage join would repeat
each ticket for every monthly usage row, so usage must be aggregated before
it reaches ticket grain — that aggregation happens in stage 3, but it isn't
safe to do until this stage confirms the keys underneath it are clean.

## 1.1 Load and Standardise

Every later stage uses the same files and definitions. This cell reads each
CSV twice, conceptually: the `_raw` frames preserve the supplied values
untouched, while separate working frames receive safe formatting changes
only (column names, whitespace, blank-string handling).

In [1]:
from common import clean_source, save_show, ARTIFACTS, DATA, PLAN_ORDER, PRIORITY_ORDER
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

customers_raw = pd.read_csv(DATA / "customers.csv")
tickets_raw = pd.read_csv(DATA / "customer_support_tickets.csv")
usage_raw = pd.read_csv(DATA / "product_usage.csv")

customers = clean_source(customers_raw)
tickets = clean_source(tickets_raw)
usage = clean_source(usage_raw)
customers["customer_email"] = customers["customer_email"].str.lower()
tickets["customer_email"] = tickets["customer_email"].str.lower()
usage["collaborators"] = pd.to_numeric(usage["collaborators"], errors="coerce")

print(f"Loaded {len(customers):,} customers, {len(tickets):,} tickets and {len(usage):,} usage rows.")

Loaded 8,320 customers, 8,469 tickets and 42,210 usage rows.


## 1.2 Parse Dates

Dates are parsed into new columns rather than overwriting the supplied
text, so the original values stay inspectable if a parse looks wrong.

In [2]:
customers["account_created_date_parsed"] = pd.to_datetime(
    customers["account_created_date"], errors="coerce"
)
usage["month_parsed"] = pd.to_datetime(
    usage["month"], format="%Y-%m", errors="coerce"
)
tickets["first_response_time_parsed"] = pd.to_datetime(
    tickets["first_response_time"], dayfirst=True, errors="coerce"
)
tickets["time_to_resolution_parsed"] = pd.to_datetime(
    tickets["time_to_resolution"], dayfirst=True, errors="coerce"
)
print("Dates parsed.")

Dates parsed.


## 1.3 Check Missingness and Duplicate Keys

Only checks the fields this analysis actually depends on — collaboration,
plan, and priority — not every column in the file.

In [3]:
required_missing = (
    customers[["customer_id", "customer_email", "plan_type"]].isna().sum().sum()
    + usage[["customer_id", "month", "collaborators"]].isna().sum().sum()
    + tickets[["ticket_id", "customer_email", "ticket_priority"]].isna().sum().sum()
)
customer_key_duplicates = customers["customer_id"].duplicated().sum()
customer_email_duplicates = customers["customer_email"].duplicated().sum()
ticket_key_duplicates = tickets["ticket_id"].duplicated().sum()
usage_key_duplicates = usage.duplicated(["customer_id", "product", "month"]).sum()

assert customer_key_duplicates == 0
assert customer_email_duplicates == 0
assert ticket_key_duplicates == 0
assert usage_key_duplicates == 0
assert required_missing == 0
assert usage["customer_id"].isin(customers["customer_id"]).all()

print("No missing required fields. No duplicate declared keys.")

No missing required fields. No duplicate declared keys.


## 1.4 Validate the Ticket → Customer Join

A raw join can silently duplicate rows if the join key isn't actually
unique on the right-hand side. Check that explicitly rather than assuming
it from the schema.

In [4]:
ticket_match = tickets.merge(
    customers[["customer_email", "customer_id", "customer_name", "customer_age", "customer_gender"]],
    on="customer_email", how="left", validate="many_to_one",
    suffixes=("_ticket", "_master"),
)
assert len(ticket_match) == len(tickets)
assert ticket_match["customer_id"].notna().all()

identity_conflict = (
    ticket_match["customer_name_ticket"].ne(ticket_match["customer_name_master"])
    | ticket_match["customer_age_ticket"].ne(ticket_match["customer_age_master"])
    | ticket_match["customer_gender_ticket"].ne(ticket_match["customer_gender_master"])
)
print(f"Join preserves row count ({len(ticket_match):,}) and every ticket resolves to a customer.")
print(f"Identity conflicts between ticket-level and master customer fields: {identity_conflict.sum():,}")

Join preserves row count (8,469) and every ticket resolves to a customer.
Identity conflicts between ticket-level and master customer fields: 149


## 1.5 Flag (Don't Fix) Timestamp and Active-Day Anomalies

These flags matter for what NOT to do later — they are not corrected here,
because correcting them would be inventing data, and they aren't inputs to
the collaboration/plan/priority questions this pipeline actually answers.

In [5]:
invalid_time_order = tickets["time_to_resolution_parsed"] < tickets["first_response_time_parsed"]
invalid_active_days = usage["active_days"] > usage["month_parsed"].dt.days_in_month

print(f"Rows where resolution timestamp precedes first-response timestamp: {invalid_time_order.sum():,}")
print(f"Usage rows with active_days exceeding the calendar month: {invalid_active_days.sum():,}")

Rows where resolution timestamp precedes first-response timestamp: 1,365
Usage rows with active_days exceeding the calendar month: 85


## 1.6 Reconcile Row Counts and Summarise

In [6]:
row_reconciliation = pd.DataFrame({
    "table": ["Customers", "Product usage", "Support tickets"],
    "raw rows": [len(customers_raw), len(usage_raw), len(tickets_raw)],
    "cleaned rows": [len(customers), len(usage), len(tickets)],
})
row_reconciliation["rows removed"] = row_reconciliation["raw rows"] - row_reconciliation["cleaned rows"]
assert (row_reconciliation["rows removed"] == 0).all()

display(row_reconciliation.style.hide(axis="index").format({
    "raw rows": "{:,}", "cleaned rows": "{:,}", "rows removed": "{:,}"
}))
print("No records were dropped or imputed.")

table,raw rows,cleaned rows,rows removed
Customers,"8,320","8,320",0
Product usage,"42,210","42,210",0
Support tickets,"8,469","8,469",0


No records were dropped or imputed.


In [7]:
cleaning_summary = pd.DataFrame({
    "cleaning check": [
        "Headers, strings and blank values", "Emails", "Collaborator type",
        "Declared key duplicates", "Missing required analysis fields",
        "Response/resolution timestamp order", "Active days above calendar maximum",
        "Ticket/customer identity conflicts",
    ],
    "result": [
        "Standardised in working copies", "Trimmed and lower-cased", "Converted to numeric",
        f"{customer_key_duplicates + ticket_key_duplicates + usage_key_duplicates:,}",
        f"{required_missing:,}", f"{invalid_time_order.sum():,} flagged rows",
        f"{invalid_active_days.sum():,} flagged rows", f"{identity_conflict.sum():,} flagged tickets",
    ],
    "decision": [
        "Raw frames remain unchanged", "Used for complete joins", "No imputation required",
        "Keep every row", "Keep every row", "Do not calculate SLA durations",
        "Not used by this analysis", "Use the customer master for identity",
    ],
})
display(cleaning_summary.style.hide(axis="index"))

cleaning check,result,decision
"Headers, strings and blank values",Standardised in working copies,Raw frames remain unchanged
Emails,Trimmed and lower-cased,Used for complete joins
Collaborator type,Converted to numeric,No imputation required
Declared key duplicates,0,Keep every row
Missing required analysis fields,0,Keep every row
Response/resolution timestamp order,"1,365 flagged rows",Do not calculate SLA durations
Active days above calendar maximum,85 flagged rows,Not used by this analysis
Ticket/customer identity conflicts,149 flagged tickets,Use the customer master for identity


**Observed:** cleaning changes representation, not the analytical
population. All **8,320 customers, 42,210 usage rows, and 8,469 tickets**
remain. Timestamp and active-day problems are recorded, not fixed — they
aren't inputs to this pipeline's questions, so dropping those rows would
only shrink the sample without fixing anything.

**Conclusion supported:** customer and ticket rates can be reproduced at
their intended grains.

**Conclusion not supported:** the supplied fields cannot measure queue
waiting time or an SLA breach, because ticket creation time is absent.

## 1.7 Save Cleaned Tables for Downstream Stages

In [8]:
customers.to_parquet(ARTIFACTS / "customers.parquet")
tickets.to_parquet(ARTIFACTS / "tickets.parquet")
usage.to_parquet(ARTIFACTS / "usage.parquet")
print("Saved: customers.parquet, tickets.parquet, usage.parquet -> ./artifacts/")

Saved: customers.parquet, tickets.parquet, usage.parquet -> ./artifacts/
